In [4]:
import pandas as pd
import numpy as np
import os
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline


# ------------------------------------------------------------
# 1. LOAD DATASET
# ------------------------------------------------------------

df = pd.read_csv('../data/bnpl_applicants.csv')

print("Dataset Loaded Successfully!")
print("Dataset Shape:", df.shape)

print("\nFirst 5 Records:")
print(df.head())


# ------------------------------------------------------------
# 2. EXPLORATORY DATA ANALYSIS
# ------------------------------------------------------------

print("\nDataset Information:")
print(df.info())

print("\nMissing Values:")
print(df.isnull().sum())

print("\nStatistical Summary:")
print(df.describe())

print("\nDefault Distribution:")
print(df['default'].value_counts())

print("\nDefault Rate:")
print(df['default'].value_counts(normalize=True) * 100)


# ------------------------------------------------------------
# 3. FEATURE SELECTION
# ------------------------------------------------------------

FEATURES = [
    'age',
    'monthly_income',
    'bureau_score',
    'existing_loans',
    'dti_ratio',
    'tenure_months',
    'order_value',
    'on_time_pct_hist',
    'device_tenure_days',
    'app_velocity_30d',
    'city_tier'
]

X = df[FEATURES]

y = df['default']


# ------------------------------------------------------------
# 4. TRAIN / TEST SPLIT
# ------------------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("\nTrain Shape:", X_train.shape)
print("Test Shape:", X_test.shape)

print(
    f"Training Default Rate: "
    f"{y_train.mean()*100:.2f}%"
)

print(
    f"Testing Default Rate: "
    f"{y_test.mean()*100:.2f}%"
)


# ------------------------------------------------------------
# 5. PREPROCESSING PIPELINE
# ------------------------------------------------------------

pre = ColumnTransformer([

    (
        'num',
        StandardScaler(),
        [c for c in FEATURES if c != 'city_tier']
    ),

    (
        'cat',
        OneHotEncoder(drop='first'),
        ['city_tier']
    )

])


# Fit preprocessing on training data only
X_train_pre = pre.fit_transform(X_train)

# Transform test data without refitting
X_test_pre = pre.transform(X_test)


print("\nPreprocessing Completed!")

print("Training Features:", X_train_pre.shape)
print("Testing Features:", X_test_pre.shape)


# ------------------------------------------------------------
# 6. SMOTE - HANDLE CLASS IMBALANCE
# ------------------------------------------------------------

print("\nBefore SMOTE:")

print(
    "Training Default Rate:",
    round(y_train.mean() * 100, 2), "%"
)

smote = SMOTE(
    sampling_strategy=0.35,
    random_state=42
)

X_train_bal, y_train_bal = smote.fit_resample(
    X_train_pre,
    y_train
)


# ------------------------------------------------------------
# 7. AFTER SMOTE VALIDATION
# ------------------------------------------------------------

print("\nAfter SMOTE:")

print("Balanced Training Shape:", X_train_bal.shape)

print(
    "Balanced Default Rate:",
    round(y_train_bal.mean() * 100, 2), "%"
)

print("\nTest Data - Untouched:")

print("Test Shape:", X_test_pre.shape)

print(
    "Test Default Rate:",
    round(y_test.mean() * 100, 2), "%"
)


# ------------------------------------------------------------
# 8. SAVE PREPROCESSOR
# ------------------------------------------------------------

os.makedirs('../models', exist_ok=True)

joblib.dump(
    pre,
    '../models/preprocessor.pkl'
)

print("\nPreprocessor Saved Successfully!")
print("Location: ../models/preprocessor.pkl")

Dataset Loaded Successfully!
Dataset Shape: (30000, 13)

First 5 Records:
  applicant_id  age  monthly_income  bureau_score  existing_loans  dti_ratio  \
0   APP0000000   49         25796.0         720.0               1       15.7   
1   APP0000001   35         51414.0         717.0               0       30.8   
2   APP0000002   28         41074.0         663.0               1       35.3   
3   APP0000003   41         48058.0         634.0               1       40.3   
4   APP0000004   39         16176.0         763.0               1       46.5   

   tenure_months  order_value  on_time_pct_hist  device_tenure_days  \
0              6        820.0              66.7                10.0   
1              3       3515.0              88.5               198.0   
2              1       2092.0              84.5               466.0   
3              6       1634.0              82.4               834.0   
4              9       1280.0              96.1               396.0   

   app_velocity_30